# Loading Libraries and Packages 

In [1]:
import open3d as o3d
import numpy as np
import os
import random
from collections import defaultdict

# Loading Point Cloud

In [2]:
# Load point cloud
workspace_dir = "/Users/arjunmallick/Reconstruction_Projects/Main-Building-Dense-2/"
dense_dir = os.path.join(workspace_dir,"dense")
pcd = o3d.io.read_point_cloud(os.path.join(dense_dir,"fused.ply"))

print(pcd)

PointCloud with 3441841 points.


In [3]:
# Bounding box
bbox = pcd.get_axis_aligned_bounding_box()
min_bound = bbox.min_bound
max_bound = bbox.max_bound

print("Min bound:", min_bound)
print("Max bound:", max_bound)

# Extent of scene
extent = max_bound - min_bound
print("Scene extent (x, y, z):", extent)

Min bound: [-33.45365524 -53.76768494 -12.98976421]
Max bound: [ 20.94860268   6.96765327 152.72303772]
Scene extent (x, y, z): [ 54.40225792  60.73533821 165.71280193]


# Downsampling (Voxel Grid)

In [4]:
voxel_size = 0.09  # adjust based on scale (5 cm typical)

pcd_down = pcd.voxel_down_sample(voxel_size=voxel_size)

print("Original points:", len(pcd.points))
print("Downsampled points:", len(pcd_down.points))

Original points: 3441841
Downsampled points: 69582


## Remove Statistical Outliers

In [5]:
pcd_clean, ind = pcd_down.remove_statistical_outlier(nb_neighbors=20,std_ratio=1.75)
print("After cleaning:", len(pcd_clean.points))

After cleaning: 69383


In [6]:
# Bounding box
bbox = pcd_clean.get_axis_aligned_bounding_box()
min_bound = bbox.min_bound
max_bound = bbox.max_bound

print("Min bound:", min_bound)
print("Max bound:", max_bound)

# Extent of scene
extent = max_bound - min_bound
print("Scene extent (x, y, z):", extent)

Min bound: [-20.25208855 -13.37486839 -11.76125526]
Max bound: [19.29303551  4.69143963 15.73217106]
Scene extent (x, y, z): [39.54512405 18.06630802 27.49342632]


In [7]:
building_bbox = o3d.geometry.AxisAlignedBoundingBox(
    min_bound=[-10, -5, -2],   # tighten X, Y, Z
    max_bound=[10,  4, 10]
)

pcd_building = pcd_clean.crop(building_bbox)

# Bounding box
bbox = pcd_building.get_axis_aligned_bounding_box()
min_bound = bbox.min_bound
max_bound = bbox.max_bound

print("Min bound:", min_bound)
print("Max bound:", max_bound)

# Extent of scene
extent = max_bound - min_bound
print("Scene extent (x, y, z):", extent)

o3d.visualization.draw_geometries([pcd_building])

Min bound: [-9.99758434 -4.99978542 -1.99990153]
Max bound: [9.99647467 3.54653025 9.99800491]
Scene extent (x, y, z): [19.99405902  8.54631567 11.99790645]


In [8]:
pcd_building.estimate_normals(
    search_param=o3d.geometry.KDTreeSearchParamHybrid(
        radius=0.2,
        max_nn=30
    )
)

print(pcd_building.normalize_normals())
final_pcd = pcd_building

PointCloud with 37645 points.


In [9]:
def create_voxel_grid(points, voxel_size):
    """
    points: (N, 3) numpy array
    voxel_size: float
    """
    voxel_dict = defaultdict(list)

    for idx, point in enumerate(points):
        voxel_idx = tuple((point // voxel_size).astype(int))
        voxel_dict[voxel_idx].append(idx)

    return voxel_dict

In [10]:
def filter_voxels(voxel_dict, min_points=10):
    return {k: v for k, v in voxel_dict.items() if len(v) >= min_points}

In [11]:
points = np.asarray(final_pcd.points)

voxel_size_partition = 1.0  # larger than downsampling voxel
voxel_dict = create_voxel_grid(points, voxel_size_partition)

voxel_dict = filter_voxels(voxel_dict, min_points=50)

print("Number of voxels:", len(voxel_dict))

Number of voxels: 216


# RANSAC and Plane fitting

In [12]:
def fit_plane(p1, p2, p3):
    v1 = p2 - p1
    v2 = p3 - p1

    normal = np.cross(v1, v2)
    if np.linalg.norm(normal) == 0:
        return None

    normal = normal / np.linalg.norm(normal)
    d = -np.dot(normal, p1)

    return (*normal, d)

In [13]:
def point_to_plane_distance(points, plane):
    a, b, c, d = plane
    return np.abs(a*points[:,0] + b*points[:,1] + c*points[:,2] + d)

In [14]:
def ransac_plane(points, num_iterations=200, distance_threshold=0.03, min_inliers=80):
    best_plane = None
    best_inliers = []

    n_points = points.shape[0]

    if n_points < 3:
        return None, []

    for _ in range(num_iterations):
        idx = random.sample(range(n_points), 3)
        p1, p2, p3 = points[idx]

        plane = fit_plane(p1, p2, p3)
        if plane is None:
            continue

        distances = point_to_plane_distance(points, plane)
        inliers = np.where(distances < distance_threshold)[0]

        # Reject weak planes early
        if len(inliers) < min_inliers:
            continue

        # Planarity check (VERY IMPORTANT)
        pts = points[inliers]
        centroid = np.mean(pts, axis=0)
        cov = np.cov(pts - centroid, rowvar=False)
        eigenvalues = np.linalg.eigvals(cov)
        eigenvalues = np.sort(eigenvalues)

        planarity_score = eigenvalues[0] / (eigenvalues.sum() + 1e-6)

        # Reject non-planar (vegetation)
        if planarity_score > 0.02:
            continue

        if len(inliers) > len(best_inliers):
            best_plane = plane
            best_inliers = inliers

    return best_plane, best_inliers

In [15]:
def detect_planes_in_voxels(points, voxel_dict):
    all_planes = []
    all_inliers_global = []

    for voxel_idx, indices in voxel_dict.items():
        voxel_points = points[indices]

        plane, inliers = ransac_plane(
            voxel_points,
            num_iterations=200,
            distance_threshold=0.03,
            min_inliers=50
        )

        if plane is None or len(inliers) < 50:
            continue

        global_inliers = np.array(indices)[inliers]

        all_planes.append(plane)
        all_inliers_global.append(global_inliers)

    return all_planes, all_inliers_global

In [16]:
planes, plane_inliers = detect_planes_in_voxels(points, voxel_dict)

print("Detected planes:", len(planes))

Detected planes: 159


In [17]:
import open3d as o3d

def visualize_planes(pcd, plane_inliers):
    colors = np.zeros((len(pcd.points), 3))

    for i, inliers in enumerate(plane_inliers):
        color = np.random.rand(3)
        colors[inliers] = color

    pcd.colors = o3d.utility.Vector3dVector(colors)
    o3d.visualization.draw_geometries([pcd])

In [18]:
visualize_planes(final_pcd, plane_inliers)

# Filtering and Merging Outputs

## Filter Weak Planes

In [21]:
def filter_planes(planes, inliers_list, points,
                  min_size=120, max_planarity=0.02):

    filtered_planes = []
    filtered_inliers = []

    for p, inl in zip(planes, inliers_list):
        if len(inl) < min_size:
            continue

        pts = points[inl]

        centroid = np.mean(pts, axis=0)
        cov = np.cov(pts - centroid, rowvar=False)
        eigenvalues = np.linalg.eigvals(cov)
        eigenvalues = np.sort(eigenvalues)

        planarity_score = eigenvalues[0] / (eigenvalues.sum() + 1e-6)

        if planarity_score > max_planarity:
            continue

        filtered_planes.append(p)
        filtered_inliers.append(inl)

    return filtered_planes, filtered_inliers

In [22]:
planes_f, inliers_f = filter_planes(
    planes,
    plane_inliers,
    points,
    min_size=120,
    max_planarity=0.02
)

print("After filtering:", len(planes_f))

After filtering: 53


In [23]:
visualize_planes(final_pcd, inliers_f)

## Merge Similar Planes

In [24]:
import numpy as np

def normalize_plane(plane):
    a, b, c, d = plane
    norm = np.linalg.norm([a, b, c])
    return np.array([a, b, c, d]) / norm


def normal_similarity(p1, p2):
    return np.abs(np.dot(p1[:3], p2[:3]))


def plane_offset_distance(p1, p2):
    return abs(p1[3] - p2[3])

In [25]:
def merge_planes(planes, inliers_list, points,
                 normal_thresh=0.92,
                 dist_thresh=0.1):

    planes = [normalize_plane(p) for p in planes]

    used = [False] * len(planes)
    merged_planes = []
    merged_inliers = []

    for i in range(len(planes)):
        if used[i]:
            continue

        group = [i]
        used[i] = True

        for j in range(i + 1, len(planes)):
            if used[j]:
                continue

            n_sim = np.abs(np.dot(planes[i][:3], planes[j][:3]))

            if n_sim < normal_thresh:
                continue

            # IMPORTANT: use centroid distance instead of d
            pts_i = points[inliers_list[i]]
            pts_j = points[inliers_list[j]]

            centroid_i = np.mean(pts_i, axis=0)
            centroid_j = np.mean(pts_j, axis=0)

            dist = np.linalg.norm(centroid_i - centroid_j)

            if dist < dist_thresh:
                group.append(j)
                used[j] = True

        merged = np.concatenate([inliers_list[g] for g in group])

        merged_planes.append(planes[i])
        merged_inliers.append(np.unique(merged))

    return merged_planes, merged_inliers

In [34]:
points = np.asarray(final_pcd.points)

merged_planes, merged_inliers = merge_planes(
    planes_f,
    inliers_f,
    points,
    normal_thresh=0.90,
    dist_thresh=0.25   # slightly relaxed for your dataset
)

print("Planes before merging:", len(planes_f))
print("Planes after merging:", len(merged_planes))

Planes before merging: 53
Planes after merging: 53


In [28]:
visualize_planes(final_pcd, merged_inliers)

## Refine Planes (Re-fit Properly)

In [35]:
def refine_plane(points):
    centroid = np.mean(points, axis=0)
    cov = np.cov(points - centroid, rowvar=False)

    _, _, vh = np.linalg.svd(cov)
    normal = vh[-1]

    d = -np.dot(normal, centroid)
    return (*normal, d)

In [36]:
refined_planes = []

for inliers in merged_inliers:
    pts = points[inliers]
    refined_planes.append(refine_plane(pts))

In [37]:
def select_top_planes(planes, inliers_list, top_k=20):
    sizes = [len(i) for i in inliers_list]
    idx = np.argsort(sizes)[::-1][:top_k]

    return [planes[i] for i in idx], [inliers_list[i] for i in idx]

In [38]:
final_planes, final_inliers = select_top_planes(
    refined_planes,
    merged_inliers,
    top_k=25
)

print("Final planes:", len(final_planes))

Final planes: 25


In [49]:
import open3d as o3d
import numpy as np

def visualize_point_cloud(pcd, point_size=2.0):
    vis = o3d.visualization.Visualizer()
    vis.create_window()
    vis.add_geometry(pcd)

    opt = vis.get_render_option()
    opt.point_size = point_size

    vis.run()
    vis.destroy_window()

In [50]:
import open3d as o3d

def visualize_planes(pcd, inliers_list):
    colors = np.zeros((len(pcd.points), 3))

    for inliers in inliers_list:
        color = np.random.rand(3)
        colors[inliers] = color

    pcd.colors = o3d.utility.Vector3dVector(colors)
    o3d.visualization.draw_geometries([pcd])

In [105]:
visualize_planes(pcd_building, final_inliers)

In [51]:
def visualize_kept_vs_removed(pcd, inliers_list):
    all_inliers = np.concatenate(inliers_list)
    mask = np.zeros(len(pcd.points), dtype=bool)
    mask[all_inliers] = True

    colors = np.zeros((len(pcd.points), 3))

    # kept points → green
    colors[mask] = [1, 0, 0]

    # removed points → red
    colors[~mask] = [0, 0, 1]

    pcd_vis = o3d.geometry.PointCloud()
    pcd_vis.points = pcd.points
    pcd_vis.colors = o3d.utility.Vector3dVector(colors)

    visualize_point_cloud(pcd_vis)

In [63]:
visualize_kept_vs_removed(pcd_building, final_inliers)

In [52]:
def visualize_single_plane(pcd, inliers):
    colors = np.zeros((len(pcd.points), 3))
    colors[inliers] = [1, 0, 0]  # highlight plane

    pcd_vis = o3d.geometry.PointCloud()
    pcd_vis.points = pcd.points
    pcd_vis.colors = o3d.utility.Vector3dVector(colors)

    visualize_point_cloud(pcd_vis)

In [65]:
visualize_single_plane(pcd_building, final_inliers[0])

In [39]:
def visualize_only_planes(pcd, inliers_list):
    import numpy as np
    import open3d as o3d

    # Combine all inliers
    all_inliers = np.concatenate(inliers_list)

    # Remove duplicates
    all_inliers = np.unique(all_inliers)

    # Extract only planar points
    pcd_planes = pcd.select_by_index(all_inliers)

    # Assign colors per plane
    colors = np.zeros((len(pcd_planes.points), 3))

    # Map original indices to new indices
    index_map = {idx: i for i, idx in enumerate(all_inliers)}

    for inliers in inliers_list:
        color = np.random.rand(3)

        for idx in inliers:
            if idx in index_map:
                colors[index_map[idx]] = color

    pcd_planes.colors = o3d.utility.Vector3dVector(colors)

    o3d.visualization.draw_geometries([pcd_planes])

In [40]:
visualize_only_planes(pcd_building, final_inliers)

# DBSCAN Merging

In [41]:
import numpy as np

def normalize_plane(plane):
    a, b, c, d = plane
    norm = np.linalg.norm([a, b, c])
    return np.array([a, b, c, d]) / norm


def build_plane_features(planes, inliers_list, points):
    features = []

    for p, inl in zip(planes, inliers_list):
        p = normalize_plane(p)
        n = p[:3]

        centroid = np.mean(points[inl], axis=0)

        feat = np.hstack([n, centroid])
        features.append(feat)

    return np.array(features)

In [42]:
from sklearn.cluster import DBSCAN

def cluster_planes_dbscan(planes, inliers_list, points,
                         eps=0.5,
                         min_samples=2):

    features = build_plane_features(planes, inliers_list, points)

    clustering = DBSCAN(eps=eps, min_samples=min_samples).fit(features)
    labels = clustering.labels_

    clusters = {}
    for i, label in enumerate(labels):
        if label == -1:
            continue

        clusters.setdefault(label, []).append(i)

    merged_planes = []
    merged_inliers = []

    for cluster in clusters.values():
        merged = np.concatenate([inliers_list[i] for i in cluster])

        merged_planes.append(planes[cluster[0]])
        merged_inliers.append(np.unique(merged))

    return merged_planes, merged_inliers

In [43]:
def refine_plane(points):
    centroid = np.mean(points, axis=0)
    cov = np.cov(points - centroid, rowvar=False)

    _, _, vh = np.linalg.svd(cov)
    normal = vh[-1]

    d = -np.dot(normal, centroid)
    return (*normal, d)

In [44]:
refined_planes = []

for inliers in merged_inliers:
    pts = points[inliers]
    refined_planes.append(refine_plane(pts))

In [45]:
def select_top_planes(planes, inliers_list, top_k=20):
    sizes = [len(i) for i in inliers_list]
    idx = np.argsort(sizes)[::-1][:top_k]

    return [planes[i] for i in idx], [inliers_list[i] for i in idx]

In [46]:
import open3d as o3d

def visualize_only_planes(pcd, inliers_list):
    import numpy as np

    all_inliers = np.unique(np.concatenate(inliers_list))
    pcd_planes = pcd.select_by_index(all_inliers)

    colors = np.zeros((len(pcd_planes.points), 3))

    index_map = {idx: i for i, idx in enumerate(all_inliers)}

    for inliers in inliers_list:
        color = np.random.rand(3)

        for idx in inliers:
            if idx in index_map:
                colors[index_map[idx]] = color

    pcd_planes.colors = o3d.utility.Vector3dVector(colors)
    o3d.visualization.draw_geometries([pcd_planes])

In [50]:
# Step 1: Filter small planes
planes_f, inliers_f = filter_planes(
    planes,
    plane_inliers,
    points,
    min_size=120,
    max_planarity=0.02
)

# Step 2: DBSCAN merging
merged_planes, merged_inliers = cluster_planes_dbscan(
    planes_f,
    inliers_f,
    points,
    eps=0.2,          # key param
    min_samples=3
)

print("After DBSCAN:", len(merged_planes))

After DBSCAN: 0


In [63]:
# Step 3: Refine
points = np.asarray(pcd_building.points)
refined_planes = [refine_plane(points[inl]) for inl in merged_inliers]

# Step 4: Select top planes
final_planes, final_inliers = select_top_planes(
    refined_planes,
    merged_inliers,
    top_k=20
)

print("Final planes:", len(final_planes))

Final planes: 3


In [64]:
# Step 5: Visualize
visualize_only_planes(pcd_building, final_inliers)